# AutoSort Evaluation Pipeline

Main steps:
1. Load data (same as training)
2. Prepare evaluation data for each time segment
3. Load models and evaluate for each clique and time segment


In [5]:
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')

import torch
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from utils_clique import (
    prepare_training_data,
    evaluate_autosort_model,
    build_sliding_cliques,
    CliqueInfo,
    compute_valid_channels,
    visualize_umap_features,
    SimpleAutoSort,
    SimpleWaveformLoader
)


In [6]:
# Load data (same as training)
recording_path = "/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s.h5"
spike_inf_path = "/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/clique/spike_inf.tsv"
neuron_inf_path = "/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/clique/neuron_inf.pkl"

# Load recording and sorting using MEArec
recording, sorting = se.read_mearec(recording_path)

# Get probe from recording
probe = recording.get_probe()
if probe is None:
    raise ValueError("Recording does not have probe information")

# Load clique information (saved during training)
base_save_dir = "/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/autosort_input/"
clique_info_path = Path(base_save_dir) / "clique_info.pkl"

if clique_info_path.exists():
    with open(clique_info_path, 'rb') as f:
        clique_info = pickle.load(f)
    cliques = clique_info['cliques']
    print(f"Loaded {len(cliques)} cliques from {clique_info_path}")
else:
    # Build cliques from probe (same as training)
    cliques = build_sliding_cliques(
        probe,
        clique_size=49,
        min_size=25,
        min_overlap=18,
        target_groups=12,
    )
    print(f"Built {len(cliques)} cliques")

# Preprocess recording (same as training)
recording_f = recording.rename_channels(range(384))

# Load GT data
spike_inf = None
neuron_inf = None

if Path(spike_inf_path).exists():
    spike_inf = pd.read_csv(spike_inf_path, sep='\t', index_col=0)
else:
    raise ValueError(f"spike_inf not found at {spike_inf_path}")

if Path(neuron_inf_path).exists():
    with open(neuron_inf_path, 'rb') as f:
        neuron_inf = pickle.load(f)
else:
    raise ValueError(f"neuron_inf.pkl not found at {neuron_inf_path}")

print(f"\nRecording loaded successfully")
print(f"Sampling rate: {recording_f.get_sampling_frequency()} Hz")
print(f"Number of channels: {recording_f.get_num_channels()}")
print(f"Recording duration: {recording_f.get_num_samples() / recording_f.get_sampling_frequency():.2f} seconds")


Loaded 12 cliques from /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/autosort_input/clique_info.pkl

Recording loaded successfully
Sampling rate: 10000.0 Hz
Number of channels: 384
Recording duration: 3600.00 seconds


## Step 1: Define Evaluation Time Segments


In [7]:
# Define evaluation time segments (in seconds)
# Each segment is treated as an independent recording
eval_time_segments = [
    (600, 1200),   # Segment 0: 600-1200 seconds
    (1200, 1800),  # Segment 1: 1200-1800 seconds
    (1800, 2400),  # Segment 2: 1800-2400 seconds
    (2400, 3000),  # Segment 3: 2400-3000 seconds
    (3000, 3600),  # Segment 4: 3000-3600 seconds
]

print(f"Evaluation time segments:")
for seg_id, (start_time, end_time) in enumerate(eval_time_segments):
    print(f"  Segment {seg_id}: {start_time}-{end_time} seconds ({end_time - start_time} seconds)")


Evaluation time segments:
  Segment 0: 600-1200 seconds (600 seconds)
  Segment 1: 1200-1800 seconds (600 seconds)
  Segment 2: 1800-2400 seconds (600 seconds)
  Segment 3: 2400-3000 seconds (600 seconds)
  Segment 4: 3000-3600 seconds (600 seconds)


## Step 2: Prepare Evaluation Data for Each Time Segment


## Step 2.5: Load Pre-prepared Evaluation Data


In [8]:
# Load pre-prepared evaluation data from disk
# Scan all clique directories to find available eval_segment data

import re
from pathlib import Path

# Define evaluation time segments (should match the segments used during data preparation)
eval_time_segments = [
    (600, 1200),   # Segment 0: 600-1200 seconds
    (1200, 1800),  # Segment 1: 1200-1800 seconds
    (1800, 2400),  # Segment 2: 1800-2400 seconds
    (2400, 3000),  # Segment 3: 2400-3000 seconds
    (3000, 3600),  # Segment 4: 3000-3600 seconds
]

print(f"Scanning for pre-prepared evaluation data in: {base_save_dir}")
print(f"Expected evaluation time segments:")
for seg_id, (start_time, end_time) in enumerate(eval_time_segments):
    print(f"  Segment {seg_id}: {start_time}-{end_time} seconds ({end_time - start_time} seconds)")

# Scan for available eval data
all_eval_data_dirs = {}  # {segment_id: {clique_id: eval_data_dir}}

base_path = Path(base_save_dir)

# Find all clique directories
clique_dirs = sorted([d for d in base_path.iterdir() if d.is_dir() and d.name.startswith('clique_')])

print(f"\nFound {len(clique_dirs)} clique directories")

for clique_dir in clique_dirs:
    # Extract clique_id from directory name (e.g., "clique_00" -> 0)
    clique_match = re.match(r'clique_(\d+)', clique_dir.name)
    if not clique_match:
        continue
    clique_id = int(clique_match.group(1))
    
    # Find all eval_segment directories
    eval_segment_dirs = sorted([d for d in clique_dir.iterdir() 
                                if d.is_dir() and d.name.startswith('eval_segment_')])
    
    for eval_segment_dir in eval_segment_dirs:
        # Extract segment_id from directory name (e.g., "eval_segment_00" -> 0)
        segment_match = re.match(r'eval_segment_(\d+)', eval_segment_dir.name)
        if not segment_match:
            continue
        segment_id = int(segment_match.group(1))
        
        # Check if train_data directory exists and has required files
        train_data_dir = eval_segment_dir / "train_data"
        if not train_data_dir.exists():
            continue
        
        # Check for required files
        required_files = [
            'X_waveform.pkl',
            'X_spiketrain_time.pkl',
            'Y_spike_id.pkl',
            'Y_spike_id_noise.pkl',
            'neuron_mapping.pkl'
        ]
        
        all_files_exist = all((train_data_dir / fname).exists() for fname in required_files)
        
        if all_files_exist:
            if segment_id not in all_eval_data_dirs:
                all_eval_data_dirs[segment_id] = {}
            all_eval_data_dirs[segment_id][clique_id] = str(train_data_dir)
            print(f"  Found: Clique {clique_id:02d} - Segment {segment_id:02d} -> {train_data_dir}")

# Print summary
print(f"\n{'='*80}")
print(f"Summary of loaded evaluation data:")
print(f"{'='*80}")
for seg_id in sorted(all_eval_data_dirs.keys()):
    cliques_in_segment = sorted(all_eval_data_dirs[seg_id].keys())
    print(f"Segment {seg_id}: {len(cliques_in_segment)} cliques (cliques: {cliques_in_segment})")

total_segments = len(all_eval_data_dirs)
total_clique_segments = sum(len(cliques) for cliques in all_eval_data_dirs.values())
print(f"\nTotal: {total_segments} segments, {total_clique_segments} clique-segment pairs")
print(f"{'='*80}")


Scanning for pre-prepared evaluation data in: /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/autosort_input/
Expected evaluation time segments:
  Segment 0: 600-1200 seconds (600 seconds)
  Segment 1: 1200-1800 seconds (600 seconds)
  Segment 2: 1800-2400 seconds (600 seconds)
  Segment 3: 2400-3000 seconds (600 seconds)
  Segment 4: 3000-3600 seconds (600 seconds)

Found 12 clique directories
  Found: Clique 00 - Segment 00 -> /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/autosort_input/clique_00/eval_segment_00/train_data
  Found: Clique 00 - Segment 01 -> /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/autosort_input/clique_00/eval_segment_01/train_data
  Found: Clique 00 -

In [ ]:
# # Detection parameters (same as training)
# detection_params = {
#     'thr_min': 2.7,
#     'thr_max': 15,
#     'distance': 4,
#     'ch_max_simul_firing': 8,
#     'wlen': 6,
#     'prominence': 10,
# }

# # Waveform window parameters (same as training)
# window_params = {
#     'left_sample': 10,
#     'right_sample': 20,
# }

# # Process each time segment and clique
# sampling_rate = recording_f.get_sampling_frequency()
# all_eval_data_dirs = {}  # {segment_id: {clique_id: eval_data_dir}}

# for seg_id, (start_time, end_time) in enumerate(eval_time_segments):
#     print(f"\n{'='*80}")
#     print(f"Processing Time Segment {seg_id}: {start_time}-{end_time} seconds")
#     print(f"{'='*80}")
    
#     # Convert time to samples
#     start_sample = int(start_time * sampling_rate)
#     end_sample = int(end_time * sampling_rate)
#     duration_seconds = end_time - start_time
    
#     # Extract time segment from recording
#     recording_segment = recording_f.frame_slice(start_frame=start_sample, end_frame=end_sample)
    
#     # Filter spike_inf to this time segment
#     spike_inf_segment = spike_inf[
#         (spike_inf['time'] >= start_sample) & (spike_inf['time'] < end_sample)
#     ].copy()
#     # Adjust time to be relative to segment start
#     spike_inf_segment['time'] = spike_inf_segment['time'] - start_sample
    
#     print(f"Segment {seg_id}:")
#     print(f"  Recording samples: {start_sample}-{end_sample} ({duration_seconds:.2f} seconds)")
#     print(f"  GT spike count: {len(spike_inf_segment)}")
    
#     all_eval_data_dirs[seg_id] = {}
    
#     # Process each clique
#     for clique_id, clique in enumerate(cliques):
#         # Create clique-specific save directory for this segment
#         save_dir = Path(base_save_dir) / f"clique_{clique_id:02d}" / f"eval_segment_{seg_id:02d}"
#         save_dir.mkdir(parents=True, exist_ok=True)
        
#         # Get clique channels (probe channel indices, 0-based)
#         clique_channels = set(clique.device_channel_indices)
        
#         # Create recording_clique by selecting clique channels from segment
#         clique_channel_ids = list(clique_channels)
#         recording_clique = recording_segment.select_channels(channel_ids=clique_channel_ids)
        
#         # Filter neurons based on channel_id: only keep neurons whose all channel_id are in clique
#         neuron_ids_in_clique = []
#         for idx, row in neuron_inf.iterrows():
#             neuron_channel_ids = row.get('channel_id', [])
#             # Handle different formats: list, string representation of list, etc.
#             if isinstance(neuron_channel_ids, str):
#                 import ast
#                 try:
#                     neuron_channel_ids = ast.literal_eval(neuron_channel_ids)
#                 except:
#                     neuron_channel_ids = []
#             elif not isinstance(neuron_channel_ids, (list, tuple, np.ndarray)):
#                 neuron_channel_ids = []
            
#             # Convert to set for easy comparison
#             neuron_channel_set = set(neuron_channel_ids)
            
#             # Check if all channels of this neuron are in the clique
#             if len(neuron_channel_set) > 0 and neuron_channel_set.issubset(clique_channels):
#                 neuron_ids_in_clique.append(row['Neuron'])
        
#         # Filter neuron_inf
#         neuron_inf_clique = neuron_inf[neuron_inf['Neuron'].isin(neuron_ids_in_clique)].copy()
        
#         # Filter spike_inf to clusters/neurons within clique
#         cluster_ids_in_clique = neuron_ids_in_clique.copy()
        
#         if 'neuron' in spike_inf_segment.columns:
#             spike_inf_clique = spike_inf_segment[spike_inf_segment['neuron'].isin(cluster_ids_in_clique)].copy()
#         elif 'cluster' in spike_inf_segment.columns:
#             spike_inf_clique = spike_inf_segment[spike_inf_segment['cluster'].isin(cluster_ids_in_clique)].copy()
#         else:
#             raise ValueError(f"spike_inf must have either 'neuron' or 'cluster' column")
        
#         if len(spike_inf_clique) == 0:
#             print(f"  Clique {clique_id}: No spikes in this segment, skipping...")
#             continue
        
#         # Compute valid_channels: only channels that have GT neurons
#         try:
#             valid_channels = compute_valid_channels(
#                 recording_clique=recording_clique,
#                 neuron_inf_clique=neuron_inf_clique,
#             )
#         except Exception as e:
#             print(f"  Warning: Failed to compute valid channels for Clique {clique_id}: {e}")
#             valid_channels = None
        
#         # Prepare evaluation data for this clique and segment
#         print(f"  Clique {clique_id}: Preparing evaluation data...")
#         eval_data_dir = prepare_training_data(
#             recording_f=recording_clique,
#             spike_inf=spike_inf_clique,
#             neuron_inf=neuron_inf_clique,
#             save_dir=str(save_dir),
#             duration_seconds=duration_seconds,
#             valid_channels=valid_channels,
#             **detection_params,
#             **window_params
#         )
        
#         all_eval_data_dirs[seg_id][clique_id] = eval_data_dir
#         print(f"  Clique {clique_id}: Evaluation data saved to {eval_data_dir}")


## Step 3: Load Models and Evaluate for Each Clique and Time Segment


In [10]:
%%capture output
# Evaluation parameters
evaluation_params = {
    'batch_size': 512,
    'left_sample': 10,
    'right_sample': 20,
}

# Number of training runs per clique
n_runs = 5

# Store all evaluation results
all_results = {}  # {segment_id: {clique_id: {run_id: results}}}

for seg_id in range(len(eval_time_segments)):
    print(f"\n{'='*80}")
    print(f"Evaluating Time Segment {seg_id}")
    print(f"{'='*80}")
    
    if seg_id not in all_eval_data_dirs:
        print(f"No evaluation data for segment {seg_id}, skipping...")
        continue
    
    all_results[seg_id] = {}
    
    for clique_id in range(len(cliques)):
        if clique_id not in all_eval_data_dirs[seg_id]:
            print(f"  Clique {clique_id}: No evaluation data, skipping...")
            continue
        
        print(f"\n{'-'*60}")
        print(f"Clique {clique_id:02d} - Segment {seg_id:02d}")
        print(f"{'-'*60}")
        
        eval_data_dir = all_eval_data_dirs[seg_id][clique_id]
        
        # Get number of channels for this clique
        n_channels = len(cliques[clique_id].device_channel_indices)
        
        all_results[seg_id][clique_id] = {}
        
        # Evaluate each training run
        for run_id in range(1, n_runs + 1):
            print(f"\n  Evaluating run {run_id}/{n_runs}...")
            
            # Model save directory for this clique and run
            model_save_dir = Path(base_save_dir) / f"clique_{clique_id:02d}" / "model_save" / f"run_{run_id}"
            
            if not model_save_dir.exists():
                print(f"    Model not found at {model_save_dir}, skipping...")
                continue
            
            # Evaluate model
            try:
                results = evaluate_autosort_model(
                    train_data_dir=eval_data_dir,
                    model_save_dir=str(model_save_dir) + "/",
                    n_channels=n_channels,
                    **evaluation_params,
                    save_results=True,
                    results_save_dir=str(model_save_dir) + f"/eval_segment_{seg_id:02d}/",
                )
                
                all_results[seg_id][clique_id][run_id] = results
                
                print(f"    Run {run_id} evaluation completed!")
                print(f"      Noise accuracy: {results['noise_accuracy']:.4f}")
                if len(results['gt_units']) > 0:
                    print(f"      Unit accuracy: {results['unit_accuracy']:.4f}")
                    print(f"      Unit F1 score: {results['unit_f1_score']:.4f}")
            except Exception as e:
                print(f"    Error evaluating run {run_id}: {e}")
                import traceback
                traceback.print_exc()

print(f"\n{'='*80}")
print(f"All evaluations completed!")
print(f"{'='*80}")


: 

## Step 5: UMAP Visualization for Noise Detection and Classification


In [ ]:
# Load neuron mapping to create results_df for UMAP visualization
# We'll create UMAP visualizations for selected segments, cliques, and runs

# Select which results to visualize (you can modify this)
visualize_segments = [0, 2, 4]  # Visualize segments 0, 2, 4
visualize_cliques = [0, 1, 2]  # Visualize cliques 0, 1, 2
visualize_runs = [1]  # Visualize run 1 for each clique

# Neuron color mapping (you can customize this)
neuron_inf_color = ["#a74a5b", "#d64158", "#e28572", "#d6522c",
                    "#a5572c", "#da9131", "#d6a46a", "#8c6d2c",
                    "#c0ab39", "#6d7821", "#9cb835", "#67733a",
                    "#9eb56c", "#4c902f", "#61c350", "#418348",
                    "#54c083", "#338b70", "#51c6c0", "#609dd8",
                    "#6365ab", "#636edd", "#a85aca", "#c590d9",
                    "#9c4d88", "#d5449a", "#e280a9"]

for seg_id in visualize_segments:
    if seg_id not in all_results:
        continue
    
    start_time, end_time = eval_time_segments[seg_id]
    print(f"\n{'='*80}")
    print(f"UMAP Visualization for Segment {seg_id} ({start_time}-{end_time} seconds)")
    print(f"{'='*80}")
    
    for clique_id in visualize_cliques:
        if clique_id not in all_results[seg_id]:
            continue
        
        for run_id in visualize_runs:
            if run_id not in all_results[seg_id][clique_id]:
                continue
            
            print(f"\n{'-'*60}")
            print(f"Clique {clique_id:02d} - Segment {seg_id:02d} - Run {run_id}")
            print(f"{'-'*60}")
            
            results = all_results[seg_id][clique_id][run_id]
            
            # Get way3 features
            way3_features_100d = results.get('way3_features_100d', np.array([]))
            way3_features_30d = results.get('way3_features_30d', np.array([]))
            
            if len(way3_features_100d) == 0 and len(way3_features_30d) == 0:
                print("  No way3 features available, skipping UMAP visualization...")
                continue
            
            # Load neuron mapping from evaluation data
            eval_data_dir = all_eval_data_dirs[seg_id][clique_id]
            neuron_mapping_path = Path(eval_data_dir) / "neuron_mapping.pkl"
            
            if not neuron_mapping_path.exists():
                print(f"  Neuron mapping not found at {neuron_mapping_path}, skipping...")
                continue
            
            with open(neuron_mapping_path, 'rb') as f:
                neuron_mapping = pickle.load(f)
            
            train_neuron_list = neuron_mapping['id_to_neuron'].values().tolist()
            
            # Create results_df for label classification visualization
            # We need to map unit predictions and GT units to neuron names
            gt_noise = results['gt_noise']
            pred_noise = results['noise_predictions']
            gt_units = results['gt_units']
            pred_units = results['unit_predictions']
            
            # Create results_df: only for spikes that passed noise classifier
            # way3_features_30d corresponds to spikes where gt_noise == 1 (non-noise samples)
            # gt_units and pred_units also correspond to these same spikes
            # So we can directly map them
            
            results_df_list = []
            
            # way3_features_30d length should match gt_units length (both are for non-noise samples)
            n_spikes_passed = len(way3_features_30d)
            
            if len(gt_units) > 0 and n_spikes_passed > 0:
                # Ensure we don't exceed the length of gt_units
                n_samples = min(n_spikes_passed, len(gt_units))
                
                for i in range(n_samples):
                    gt_unit_id = gt_units[i]
                    pred_unit_id = pred_units[i] if i < len(pred_units) else -1
                    
                    # Map unit IDs to neuron names
                    gt_label = neuron_mapping['id_to_neuron'].get(gt_unit_id, 'unmatch')
                    pred_label = neuron_mapping['id_to_neuron'].get(pred_unit_id, 'unmatch')
                    
                    results_df_list.append({
                        'gt_label': gt_label,
                        'predicted_label': pred_label,
                    })
            
            results_df = pd.DataFrame(results_df_list) if len(results_df_list) > 0 else pd.DataFrame()
            
            # Create neuron color mapping
            neuron_color_dict = {}
            for i, neuron in enumerate(train_neuron_list):
                if i < len(neuron_inf_color):
                    neuron_color_dict[neuron] = neuron_inf_color[i]
                else:
                    import matplotlib.pyplot as plt
                    cmap = plt.cm.get_cmap('tab20')
                    neuron_color_dict[neuron] = cmap(i % 20)
            
            # Generate UMAP visualization
            try:
                figs = visualize_umap_features(
                    way3_features_100d=way3_features_100d,
                    way3_features_30d=way3_features_30d,
                    results_df=results_df,
                    train_neuron_list=train_neuron_list,
                    noise_gt_labels=gt_noise,
                    noise_pred_labels=pred_noise,
                    neuron_inf_color=neuron_color_dict,
                    n_samples=50000,
                    random_state=42
                )
                
                # Save figures
                umap_save_dir = Path(base_save_dir) / f"clique_{clique_id:02d}" / "model_save" / f"run_{run_id}" / f"eval_segment_{seg_id:02d}" / "umap"
                umap_save_dir.mkdir(parents=True, exist_ok=True)
                
                figure_names = [
                    'noise_detection_gt',
                    'noise_detection_pred',
                    'label_classification_gt',
                    'label_classification_pred'
                ]
                
                for fig, name in zip(figs, figure_names):
                    if fig is not None:
                        save_path = umap_save_dir / f"{name}.pdf"
                        fig.savefig(save_path, dpi=300, bbox_inches='tight')
                        print(f"  Saved {name} to {save_path}")
                        plt.close(fig)
                
                print(f"  UMAP visualization completed!")
                
            except Exception as e:
                print(f"  Error generating UMAP visualization: {e}")
                import traceback
                traceback.print_exc()

print(f"\n{'='*80}")
print(f"UMAP visualization completed!")
print(f"{'='*80}")


In [ ]:
import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from utils_clique import detect_spike

def process_single_clique_timing(
    recording_clique,
    autosort_model,
    calibration_results,
    start_frame,
    time_window_seconds,
    detection_params,
    window_params,
    device=None,
):
    """
    Process a single clique time window and measure processing time.
    Returns processing time in seconds (from threshold detection to output results).
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    left_sample = window_params['left_sample']
    right_sample = window_params['right_sample']
    window_size = left_sample + right_sample
    sampling_frequency = recording_clique.get_sampling_frequency()
    
    # Get models and mapping from calibration stage
    kmeans_model = calibration_results['kmeans_model']
    pca_model = calibration_results['pca_model']
    cluster_to_neuron_mapping = calibration_results['cluster_to_neuron_mapping']
    
    # Start timing: from threshold detection
    start_time = time.time()
    
    # 1. Load current window data
    window_frames = int(time_window_seconds * sampling_frequency)
    end_frame = min(start_frame + window_frames, recording_clique.get_num_samples())
    traces = recording_clique.get_traces(start_frame=start_frame, end_frame=end_frame)
    if traces.shape[0] > traces.shape[1] and traces.shape[0] > 100:
        traces = traces.T
    traces = traces.astype(np.float32)
    
    # 2. Threshold detection
    trace0_car = traces.T  # (n_timepoints, n_channels)
    spikes = detect_spike(trace0_car, **detection_params)
    spike_coords = np.argwhere(spikes == 1)  # (n_spikes, 2) [time, channel]
    
    if len(spike_coords) == 0:
        return 0.0
    
    # 3. Extract waveforms and filter boundaries
    waveforms = []
    spike_times = []
    spike_channels = []
    
    for time_idx, channel_idx in spike_coords:
        global_time_idx = start_frame + time_idx
        local_start = time_idx - left_sample
        local_end = time_idx + right_sample
        
        if local_start < 0 or local_end > trace0_car.shape[0]:
            continue
        if local_end - local_start != window_size:
            continue
        
        waveform = traces[:, local_start:local_end]  # (n_channels, window_size)
        waveforms.append(waveform)
        spike_times.append(global_time_idx)
        spike_channels.append(channel_idx)
    
    if len(waveforms) == 0:
        return 0.0
    
    waveforms = np.array(waveforms)  # (n_spikes, n_channels, window_size)
    
    # 4. Pass through noise classifier, classified as spikes
    batch_size = 512
    n_spikes = len(waveforms)
    way3_features_list = []
    way3_spike_indices = []
    
    autosort_model.eval()
    with torch.no_grad():
        for i in range(0, n_spikes, batch_size):
            batch_end = min(i + batch_size, n_spikes)
            batch_waveforms = waveforms[i:batch_end]
            batch_channels = spike_channels[i:batch_end]
            
            batch_single_waveforms = []
            batch_multi_waveforms = []
            
            for wf, ch in zip(batch_waveforms, batch_channels):
                multi_wf = wf.flatten()
                batch_multi_waveforms.append(multi_wf)
                single_wf = wf[ch, :]
                batch_single_waveforms.append(single_wf)
            
            batch_multi_waveforms = np.array(batch_multi_waveforms)
            batch_single_waveforms = np.array(batch_single_waveforms)
            
            batch_multi = torch.from_numpy(batch_multi_waveforms).float().to(device)
            batch_single = torch.from_numpy(batch_single_waveforms).float().to(device)
            
            codes = torch.cat((batch_multi, batch_single), dim=1)
            
            noise_output = autosort_model.clsfier_noise(codes)
            noise_pred = torch.argmax(noise_output, dim=1)
            
            spike_mask = noise_pred == 1
            if spike_mask.sum() > 0:
                batch_spike_indices = np.arange(i, batch_end)[spike_mask.cpu().numpy()]
                way3_spike_indices.extend(batch_spike_indices.tolist())
                
                codes_spike = codes[spike_mask]
                way3_batch = autosort_model.clsfier_label.intermediate_forward(codes_spike)
                way3_features_list.append(way3_batch.cpu().numpy())
    
    if len(way3_features_list) == 0:
        end_time = time.time()
        return end_time - start_time
    
    way3_features = np.concatenate(way3_features_list, axis=0)
    
    # 5. PCA dimensionality reduction
    way3_pca = pca_model.transform(way3_features)
    
    # 6. K-means prediction
    cluster_labels = kmeans_model.predict(way3_pca)
    
    # 7. Map to train neuron ID
    neuron_predictions = []
    for cluster_id in cluster_labels:
        if cluster_id in cluster_to_neuron_mapping:
            neuron_predictions.append(cluster_to_neuron_mapping[cluster_id])
        else:
            neuron_predictions.append('unmatch')
    
    end_time = time.time()
    return end_time - start_time


# Test parameters
window_sizes_ms = [100, 200, 300, 400, 500, 1000]  # milliseconds
n_runs_per_window = 20  # Number of segments to process for each window size

# Detection parameters
detection_params = {
    'thr_min': 2.7,
    'thr_max': 15,
    'distance': 4,
    'ch_max_simul_firing': 8,
    'wlen': 6,
    'prominence': 10,
}

# Window parameters
window_params = {
    'left_sample': 10,
    'right_sample': 20,
}

# Use first time segment for testing
test_segment_id = 0
start_time, end_time = eval_time_segments[test_segment_id]
duration_seconds = end_time - start_time
start_sample = int(start_time * sampling_rate)
end_sample = int(end_time * sampling_rate)
recording_segment = recording_f.frame_slice(start_frame=start_sample, end_frame=end_sample)

# Load models and calibration results for each clique
# We'll use run_1 for all cliques
run_id = 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

clique_models = {}
clique_calibration_results = {}

print("=" * 80)
print("Real-time Processing Time Benchmark for Cliques")
print("=" * 80)
print(f"Test segment: {test_segment_id} ({start_time}-{end_time} seconds)")
print(f"Processing window sizes to test: {window_sizes_ms} ms")
print(f"Number of runs per window size: {n_runs_per_window}")
print(f"Number of cliques: {len(cliques)}")
print(f"Loading models and calibration results for run {run_id}...")
print("=" * 80)

# Load models and calibration results for each clique
for clique_id in range(len(cliques)):
    model_save_dir = Path(base_save_dir) / f"clique_{clique_id:02d}" / "model_save" / f"run_{run_id}"
    
    if not model_save_dir.exists():
        print(f"  Warning: Model not found for clique {clique_id}, skipping...")
        continue
    
    try:
        # Load model
        n_channels = len(cliques[clique_id].device_channel_indices)
        dataset = SimpleWaveformLoader(
            train_data_dir=str(model_save_dir.parent.parent.parent / f"clique_{clique_id:02d}" / "train_data"),
            n_channels=n_channels,
            left_sample=window_params['left_sample'],
            right_sample=window_params['right_sample'],
        )
        
        autosort_model = SimpleAutoSort(
            n_channels=n_channels,
            window_size=window_params['left_sample'] + window_params['right_sample'],
            n_units=dataset.n_units,
            set_shank_id=None,
            save_dir=str(model_save_dir) + "/",
            pos_weight_noise=dataset.pos_weight_noise.to(device),
            pos_weight_label=dataset.pos_weight_label.to(device)
        )
        autosort_model.load_model()
        autosort_model.eval()
        clique_models[clique_id] = autosort_model
        
        # Load calibration results (if available)
        calibration_path = model_save_dir / "calibration_results.pkl"
        if calibration_path.exists():
            with open(calibration_path, 'rb') as f:
                clique_calibration_results[clique_id] = pickle.load(f)
        else:
            print(f"  Warning: Calibration results not found for clique {clique_id}")
    
    except Exception as e:
        print(f"  Error loading model for clique {clique_id}: {e}")
        continue

print(f"Loaded {len(clique_models)} models and {len(clique_calibration_results)} calibration results")

# Select cliques that have both model and calibration results
available_cliques = [cid for cid in clique_models.keys() if cid in clique_calibration_results]
print(f"Available cliques for testing: {available_cliques}")

if len(available_cliques) == 0:
    raise ValueError("No cliques with both model and calibration results available!")

# Initialize results
timing_results = []

# Process each window size
for window_size_ms in window_sizes_ms:
    print(f"\nTesting processing window size: {window_size_ms} ms")
    time_window_seconds = window_size_ms / 1000.0
    
    # Test single clique processing time
    single_clique_times = []
    all_cliques_serial_times = []
    all_cliques_parallel_times = []
    
    # Use available cliques
    test_cliques = [cliques[cid] for cid in available_cliques[:min(5, len(available_cliques))]]
    test_clique_ids = available_cliques[:min(5, len(available_cliques))]
    
    for run_idx in range(n_runs_per_window):
        # Random start time within segment
        random_start_time = np.random.uniform(start_time, end_time - time_window_seconds)
        start_frame = int(random_start_time * sampling_rate)
        
        # Test single clique (use first clique as example)
        if len(test_clique_ids) > 0:
            clique_id = test_clique_ids[0]
            clique = test_cliques[0]
            clique_channels = list(set(clique.device_channel_indices))
            recording_clique = recording_segment.select_channels(channel_ids=clique_channels)
            
            try:
                single_clique_time = process_single_clique_timing(
                    recording_clique=recording_clique,
                    autosort_model=clique_models[clique_id],
                    calibration_results=clique_calibration_results[clique_id],
                    start_frame=start_frame,
                    time_window_seconds=time_window_seconds,
                    detection_params=detection_params,
                    window_params=window_params,
                    device=device,
                )
                single_clique_times.append(single_clique_time)
            except Exception as e:
                print(f"    Error processing single clique: {e}")
                single_clique_times.append(np.nan)
        
        # Test serial processing of all cliques
        serial_start = time.time()
        for clique_id, clique in zip(test_clique_ids, test_cliques):
            clique_channels = list(set(clique.device_channel_indices))
            recording_clique = recording_segment.select_channels(channel_ids=clique_channels)
            try:
                process_single_clique_timing(
                    recording_clique=recording_clique,
                    autosort_model=clique_models[clique_id],
                    calibration_results=clique_calibration_results[clique_id],
                    start_frame=start_frame,
                    time_window_seconds=time_window_seconds,
                    detection_params=detection_params,
                    window_params=window_params,
                    device=device,
                )
            except Exception as e:
                print(f"    Error processing clique {clique_id} in serial: {e}")
        serial_end = time.time()
        all_cliques_serial_times.append(serial_end - serial_start)
        
        # Test parallel processing of all cliques
        # Note: For GPU processing, we need to be careful with parallel execution
        # Using threads might cause GPU contention, so we'll use sequential processing
        # but measure the time as if it were parallel (ideal case)
        # In practice, you might need to use multiple GPUs or process sequentially
        parallel_start = time.time()
        
        # For now, we'll simulate parallel by processing sequentially but measuring time
        # In a real parallel implementation, you would use ProcessPoolExecutor with separate processes
        # or ThreadPoolExecutor if GPU allows concurrent access
        def process_clique_wrapper(args):
            clique_id, clique, start_frame, time_window_seconds = args
            clique_channels = list(set(clique.device_channel_indices))
            recording_clique = recording_segment.select_channels(channel_ids=clique_channels)
            return process_single_clique_timing(
                recording_clique=recording_clique,
                autosort_model=clique_models[clique_id],
                calibration_results=clique_calibration_results[clique_id],
                start_frame=start_frame,
                time_window_seconds=time_window_seconds,
                detection_params=detection_params,
                window_params=window_params,
                device=device,
            )
        
        # Sequential processing (simulating parallel - actual parallel would require careful GPU management)
        for clique_id, clique in zip(test_clique_ids, test_cliques):
            try:
                process_clique_wrapper((clique_id, clique, start_frame, time_window_seconds))
            except Exception as e:
                print(f"    Error processing clique {clique_id} in parallel: {e}")
        
        parallel_end = time.time()
        # For true parallel, this would be max(individual_times), but we're measuring sequential
        # In practice, with proper parallelization, this should be close to max(single_clique_times)
        all_cliques_parallel_times.append(parallel_end - parallel_start)
        
        if (run_idx + 1) % 5 == 0:
            print(f"  Completed {run_idx + 1}/{n_runs_per_window} runs")
    
    # Store results
    for run_idx in range(n_runs_per_window):
        timing_results.append({
            'window_size_ms': window_size_ms,
            'run': run_idx + 1,
            'single_clique_time': single_clique_times[run_idx] if run_idx < len(single_clique_times) else np.nan,
            'all_cliques_serial_time': all_cliques_serial_times[run_idx],
            'all_cliques_parallel_time': all_cliques_parallel_times[run_idx],
        })
    
    if len(single_clique_times) > 0:
        print(f"  Single clique: {np.nanmean(single_clique_times):.4f} ± {np.nanstd(single_clique_times):.4f} seconds")
    print(f"  Serial processing (all cliques): {np.mean(all_cliques_serial_times):.4f} ± {np.std(all_cliques_serial_times):.4f} seconds")
    print(f"  Parallel processing (all cliques): {np.mean(all_cliques_parallel_times):.4f} ± {np.std(all_cliques_parallel_times):.4f} seconds")

# Convert to DataFrame
timing_df = pd.DataFrame(timing_results)

# Save results
timing_df_path = Path(base_save_dir) / "clique_processing_timing.csv"
timing_df.to_csv(timing_df_path, index=False)
print(f"\nTiming results saved to: {timing_df_path}")

print(f"\n{'='*80}")
print(f"Timing results summary:")
print(f"{'='*80}")
print(timing_df.groupby('window_size_ms').agg({
    'single_clique_time': ['mean', 'std'],
    'all_cliques_serial_time': ['mean', 'std'],
    'all_cliques_parallel_time': ['mean', 'std'],
}).round(4))


In [ ]:
# Plot bee swarm plots for three types of processing times
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Prepare data for plotting
plot_df_long = timing_df.copy()
plot_df_long['window_size_ms_str'] = plot_df_long['window_size_ms'].astype(str) + ' ms'

# Plot 1: Single clique processing time
ax1 = axes[0]
sns.swarmplot(
    data=plot_df_long,
    x='window_size_ms_str',
    y='single_clique_time',
    ax=ax1,
    size=4,
    alpha=0.7,
    color='#2E86AB'
)
sns.boxplot(
    data=plot_df_long,
    x='window_size_ms_str',
    y='single_clique_time',
    ax=ax1,
    boxprops=dict(alpha=0.3),
    showfliers=False,
    width=0.5
)
ax1.set_xlabel('Window Size', fontsize=12, fontweight='bold')
ax1.set_ylabel('Processing Time (seconds)', fontsize=12, fontweight='bold')
ax1.set_title('Single Clique Processing Time', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3, linestyle='--', axis='y')
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Plot 2: Serial processing (all cliques)
ax2 = axes[1]
sns.swarmplot(
    data=plot_df_long,
    x='window_size_ms_str',
    y='all_cliques_serial_time',
    ax=ax2,
    size=4,
    alpha=0.7,
    color='#A23B72'
)
sns.boxplot(
    data=plot_df_long,
    x='window_size_ms_str',
    y='all_cliques_serial_time',
    ax=ax2,
    boxprops=dict(alpha=0.3),
    showfliers=False,
    width=0.5
)
ax2.set_xlabel('Window Size', fontsize=12, fontweight='bold')
ax2.set_ylabel('Processing Time (seconds)', fontsize=12, fontweight='bold')
ax2.set_title('Serial Processing (All Cliques)', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, linestyle='--', axis='y')
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Plot 3: Parallel processing (all cliques)
ax3 = axes[2]
sns.swarmplot(
    data=plot_df_long,
    x='window_size_ms_str',
    y='all_cliques_parallel_time',
    ax=ax3,
    size=4,
    alpha=0.7,
    color='#F18F01'
)
sns.boxplot(
    data=plot_df_long,
    x='window_size_ms_str',
    y='all_cliques_parallel_time',
    ax=ax3,
    boxprops=dict(alpha=0.3),
    showfliers=False,
    width=0.5
)
ax3.set_xlabel('Window Size', fontsize=12, fontweight='bold')
ax3.set_ylabel('Processing Time (seconds)', fontsize=12, fontweight='bold')
ax3.set_title('Parallel Processing (All Cliques)', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3, linestyle='--', axis='y')
plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()

# Save figure
output_fig_path = Path(base_save_dir) / "clique_processing_timing_beeswarm.pdf"
plt.savefig(output_fig_path, dpi=300, bbox_inches='tight')
print(f"\nBee swarm plots saved to: {output_fig_path}")

# Also save as PNG
output_fig_path_png = Path(base_save_dir) / "clique_processing_timing_beeswarm.png"
plt.savefig(output_fig_path_png, dpi=300, bbox_inches='tight')
print(f"Bee swarm plots (PNG) saved to: {output_fig_path_png}")

plt.show()

# Print summary statistics
print(f"\n{'='*80}")
print(f"Summary Statistics:")
print(f"{'='*80}")
for window_size_ms in window_sizes_ms:
    window_data = timing_df[timing_df['window_size_ms'] == window_size_ms]
    print(f"\nWindow Size: {window_size_ms} ms")
    
    single_times = window_data['single_clique_time'].dropna()
    if len(single_times) > 0:
        print(f"  Single clique:")
        print(f"    Mean: {single_times.mean():.4f} ± {single_times.std():.4f} seconds")
        print(f"    Median: {single_times.median():.4f} seconds")
        print(f"    Range: [{single_times.min():.4f}, {single_times.max():.4f}] seconds")
    
    serial_times = window_data['all_cliques_serial_time']
    print(f"  Serial (all cliques):")
    print(f"    Mean: {serial_times.mean():.4f} ± {serial_times.std():.4f} seconds")
    print(f"    Median: {serial_times.median():.4f} seconds")
    print(f"    Range: [{serial_times.min():.4f}, {serial_times.max():.4f}] seconds")
    
    parallel_times = window_data['all_cliques_parallel_time']
    print(f"  Parallel (all cliques):")
    print(f"    Mean: {parallel_times.mean():.4f} ± {parallel_times.std():.4f} seconds")
    print(f"    Median: {parallel_times.median():.4f} seconds")
    print(f"    Range: [{parallel_times.min():.4f}, {parallel_times.max():.4f}] seconds")
    
    if len(single_times) > 0:
        speedup = serial_times.mean() / single_times.mean()
        print(f"  Serial speedup (vs single): {speedup:.2f}x")
        parallel_speedup = serial_times.mean() / parallel_times.mean()
        print(f"  Parallel speedup (vs serial): {parallel_speedup:.2f}x")
print(f"{'='*80}")
